<h1><center> Dimension Reduction </center></h1>

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from textwrap import wrap

# Change cell width to fullscreen (like colab)
# https://stackoverflow.com/questions/21971449/how-do-i-increase-the-cell-width-of-the-jupyter-ipython-notebook-in-my-browser
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>.output_result { max-width:100% !important; }</style>"))

In [ ]:
# Bold font for plots
from matplotlib import rc,rcParams
rc('font', weight='bold')
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams.update({'font.size': 18})
# rcParams['text.latex.preamble'] = [r'\usepackage{sfmath} \boldmath']

In [ ]:
def read_data(f_path, variable_list):
    dataset = pd.read_excel(f_path)
    column_names = dataset.columns

    # Change treatment title - optional
    dataset.rename(columns={'Treated (HIP/Y/N)':'Treatment Information'}, inplace=True)
    dataset.rename(columns={'Cell Spacing':'Cell Spacing (um)'}, inplace=True)
    column_names = dataset.columns
    # Change (um) to correct scientific symbol
    dataset.columns = [str.replace('(um)', '(\u03BCm)') for str in column_names]
    dataset = dataset.replace(r'\s+', np.nan, regex =True)

    # Searches selected variable's index number
    col_index = []
    for i in range(len(variable_list)): 
        col_index.append(column_names.get_loc(variable_list[i]))
    # Change (um) to correct scientific symbol
    variable_list = [str.replace('(um)', '(\u03BCm)') for str in variable_list]
    #col_index.append(column_names.get_loc(variable_list[i]))
    # Converts whole dataset to df and filters necessary columns 
    dataset = dataset[dataset.columns[col_index]]

    return dataset

In [ ]:
# Specify list of variables we want in our dataframe
col_var = ['Material',
            'Laser Power (W)',
            'Scanning Speed (mm/s)',
            'Layer Thickness (um)',
            'Hatch Spacing (um)',
            'Volumetric energy density (J/mm3)',
            'Treatment Information',
            'Yield Stress (MPa)',
            'Ultimate Tensile Stress (MPa)', 
            'Elongation (%)',
            'Consolidation (%)',
            'Average Work Hardening'
            ]

path = r'../data/am_properties.xlsx'

dataset = read_data(path, col_var)
# Filter data to specified materials
materials_list = ['316L', 'IN718', 'TI6Al4V', 'AlSi10Mg']
dataset = dataset[dataset['Material'].isin(materials_list)]
dataset = dataset.reset_index(drop=True)
dataset

In [ ]:
# Choose variables to consider (for input and output data)
input_var = ['Material',
            'Laser Power (W)',
            'Scanning Speed (mm/s)',
            'Layer Thickness (μm)',
            'Hatch Spacing (μm)']

output_var = ['Yield Stress (MPa)',
              'Average Work Hardening',
              'Elongation (%)',
              'Consolidation (%)']

# Obtains length of input and outputs
n_inputs = len(input_var)
n_outputs = len(output_var)

# Preprocessing Data

In [ ]:
# Consider materials - Uses OneHotEncoder (encoding materials and heat treatment) without Iterative Imputer (missing values)
x_df = dataset[input_var].drop(columns=['Material'])
x_df = x_df.dropna()
x_df_index = x_df.index.to_list()

# Standardize the features (mean=0 and variance=1)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_df)

total_data = len(dataset)
remain_data = x_scaled.shape[0]
dropped_data = len(dataset) - remain_data
print(" Total data = ", total_data, "\n", "Dropped data = ", dropped_data, "\n", "Remaining data = ", remain_data)

# PCA Transformation

In [ ]:
pca = PCA(n_components=2)  # Adjust the number of components according to your requirement
x_pca = pca.fit_transform(x_scaled)

print("Original shape: {}".format(str(x_scaled.shape)))
print("Reduced shape: {}".format(str(x_pca.shape)))

In [ ]:
output_df = dataset.loc[x_df_index].reset_index(drop=True)
output_df = output_df[output_var]
output_df

In [ ]:
pca_colname = ['Principal Component 1', 'Principal Component 2']
pca_df = pd.DataFrame(x_pca, columns=pca_colname)
pca_df

# Correlation Maps

In [ ]:
# Spearman's Correlation Coefficient
all_df = pd.concat([pca_df, output_df], axis=1)
all_df = all_df.corr('spearman')
all_df = all_df.drop(pca_colname)
all_df = all_df.drop(output_var, axis=1)

sns.set(font_scale=1.8)
plt.figure(figsize=(20, 6))
cmap = sns.color_palette("mako", as_cmap=True)
heatmap = sns.heatmap(all_df, vmin=-0.3, vmax=0.3, center=0,
            square=True, linewidths=.5, annot=True, fmt='.2f', cmap=cmap)

labels_x = [ '\n'.join(wrap(l, 17)) for l in output_var]
# heatmap.set_xticklabels(labels_x);
plt.xticks(rotation=45)
plt.yticks(rotation=0);

In [ ]:
# Pearson's Correlation Coefficient
all_df = pd.concat([pca_df, output_df], axis=1)
all_df = all_df.corr('pearson')
all_df = all_df.drop(pca_colname)
all_df = all_df.drop(output_var, axis=1)

sns.set(font_scale=1.8)
plt.figure(figsize=(20, 6))
cmap = sns.color_palette("mako", as_cmap=True)
heatmap = sns.heatmap(all_df, vmin=-0.3, vmax=0.3, center=0,
            square=True, linewidths=.5, annot=True, fmt='.2f', cmap=cmap)

labels_x = [ '\n'.join(wrap(l, 17)) for l in output_var]
# heatmap.set_xticklabels(labels_x);
plt.xticks(rotation=45)
plt.yticks(rotation=0);

In [ ]:
# Kendall's Correlation Coefficient
all_df = pd.concat([pca_df, output_df], axis=1)
all_df = all_df.corr('kendall')
all_df = all_df.drop(pca_colname)
all_df = all_df.drop(output_var, axis=1)

sns.set(font_scale=1.8)
plt.figure(figsize=(20, 6))
cmap = sns.color_palette("mako", as_cmap=True)
heatmap = sns.heatmap(all_df, vmin=-0.3, vmax=0.3, center=0,
            square=True, linewidths=.5, annot=True, fmt='.2f', cmap=cmap)

labels_x = [ '\n'.join(wrap(l, 17)) for l in output_var]
# heatmap.set_xticklabels(labels_x);
plt.xticks(rotation=45)
plt.yticks(rotation=0);

In [ ]:
def in_out_corr(df_x, df_y, input_params, output_params, method):
    all_df = pd.concat([df_x, df_y], axis=1)
    all_df = all_df.replace(r'\s+', np.nan, regex = True)
    all_df = all_df.corr(method)
    all_df = all_df.drop(index=input_params)
    all_df = all_df.drop(output_params, axis=1)
    return all_df

def corr_map(corr_matrices, vmin, vmax, step, c_palette='mako'):
    sns.set(font_scale=1.8)
    fig, axes = plt.subplots(1, 3, figsize=(20, 8), gridspec_kw={'width_ratios': [1, 1, 1.25]})

    for i, (corr_matrix, title) in enumerate(corr_matrices):
        cmap = sns.color_palette(c_palette, as_cmap=True)
        if i == 2:  # Only adjust color bar for the last heatmap
            heatmap = sns.heatmap(corr_matrix, center=0,
                                square=True, linewidths=.5, annot=True, fmt='.2f', cmap=cmap, ax=axes[i], vmin=vmin, vmax=vmax, cbar=True,
                                cbar_kws={"shrink": 1, "pad": 0.1})
                                
            # Adjust color bar ticks
            cbar = heatmap.collections[0].colorbar
            cbar.set_ticks(np.arange(vmin, vmax+step, step))
        else:
            heatmap = sns.heatmap(corr_matrix, center=0,
                                square=True, linewidths=.5, annot=True, fmt='.2f', cmap=cmap, ax=axes[i], vmin=vmin, vmax=vmax, cbar=False)

        labels_x = [ '\n'.join(wrap(l, 17)) for l in corr_matrices[0][0].columns.to_list()]
        labels_y = ['\n'.join(wrap(l, 17)) for l in corr_matrices[0][0].index.to_list()]

        # Replace labels
        labels_x = [str.replace('(um)', '(\u03BCm)') for str in labels_x]
        labels_x = [item.replace("(mm/s)", "(mm $\mathregular{s^{-1}}$)") for item in labels_x]
        labels_x = [item.replace("(J/mm3)", "(J $\mathregular{mm^{-3}}$)") for item in labels_x]
    
        labels_y = [str.replace('(um)', '(\u03BCm)') for str in labels_y]
        labels_y = [item.replace("(mm/s)", "(mm $\mathregular{s^{-1}}$)") for item in labels_y]
        labels_y = [item.replace("(J/mm3)", "(J $\mathregular{mm^{-3}}$)") for item in labels_y]

        heatmap.set_xticklabels(labels_x);
        heatmap.set_yticklabels(labels_y);
        axes[i].set_xticklabels(labels_x, rotation=45)
        axes[i].set_yticklabels(labels_y, rotation=0)
        axes[i].set_title(title, fontweight='bold', pad=20)

    plt.subplots_adjust(wspace=1)  # Adjusting the width space between subplots

    plt.tight_layout()
    plt.show()

In [ ]:
# Plot all 3 correlations together
pearson_corr = in_out_corr(pca_df, output_df, pca_colname, output_var, 'pearson')
spearman_corr = in_out_corr(pca_df, output_df, pca_colname, output_var, 'spearman')
kendall_corr = in_out_corr(pca_df, output_df, pca_colname, output_var, 'kendall')

# List of correlation matrices and their titles
correlation_matrices = [(pearson_corr, "Pearson's Correlation"),
                        (spearman_corr, "Spearman's Correlation"),
                        (kendall_corr, "Kendall's Tau Correlation")]

corr_map(corr_matrices=correlation_matrices, vmin=-0.3, vmax=0.3, step=0.1, c_palette='mako')

# PCA Visualisation

In [ ]:
# Filtered dataframe of used rows for PCA
filtered_df = dataset.loc[x_df_index].reset_index(drop=True)
filtered_df

In [ ]:
# Fetch hex colors
from pylab import *
cmap = cm.get_cmap('mako', 4)    # PiYG

for i in range(cmap.N):
    rgba = cmap(i)
    # rgb2hex accepts rgb or rgba
    print(matplotlib.colors.rgb2hex(rgba))

In [ ]:
from sklearn.cluster import DBSCAN
from matplotlib.colors import ListedColormap

# Extract features
X = all_df.dropna()
X = X.reset_index(drop=True)
X = X.drop(columns=['cte_min', 'cte_max'])

# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Perform PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

# Fit DBSCAN
# eps: Maximum distance between two samples for them to be considered as in the same neighborhood.
# min_samples: Number of samples (or total weight) in a neighborhood for a point to be considered as a core point - includes itself as a point.
# dbscan = DBSCAN(eps=4.5, min_samples=35)
# dbscan.fit(X_pca)

# Define custom colors for the classes
custom_colors = ['#85d9b1', '#3b589a']
cmap_custom = ListedColormap(custom_colors)

# Visualize the clusters
plt.figure(figsize=(8, 8))
# plt.scatter(X_pca[:, 0], X_pca[:, 1], c=dbscan.labels_, cmap=cmap_custom, alpha=0.5, edgecolors='black')

# Plot the points corresponding to each material
filtered_dfs = [fil_316l, fil_304l, fil_17_4PH, fil_15_5PH, fil_18Ni_300, fil_H13, fil_H11]
materials = ['316L', '304L', '17_4PH', '15_5PH', '18Ni-300', 'H13', 'H11']
colors = ['#3b4cc0', '#7092f3', '#aac7fd', '#dddddd', '#f7b89c', '#e7755b', '#b40426']
markers = ['^', 'v', 's', 'p', 'D', 'H', '*']
# Create a new dataframe with reduced dimensions
pca_df = pd.DataFrame(data=X_pca, columns=['PCA1', 'PCA2'])

for filtered_df, material, color, marker in zip(filtered_dfs, materials, colors, markers):
    filtered_indices = filtered_df.index
    filtered_points = pca_df.loc[filtered_indices]
    if marker == '*':
        plt.scatter(filtered_points['PCA1'], filtered_points['PCA2'], color=color, label=f'{material}', edgecolors='black', marker=marker, s=80)
    else:
        plt.scatter(filtered_points['PCA1'], filtered_points['PCA2'], color=color, label=f'{material}', edgecolors='black', marker=marker, s=50)

# Set plot labels and title
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.legend(fontsize=18, bbox_to_anchor=(1, 1.02), edgecolor='black', loc='upper left')
plt.show()

In [ ]:
x_df

# Explained Variance

In [ ]:
# Consider materials - Uses OneHotEncoder (encoding materials and heat treatment) without Iterative Imputer (missing values)
x_df = dataset[input_var].drop(columns=['Material'])
x_df = x_df.dropna()
x_df_index = x_df.index.to_list()

# Standardize the features (mean=0 and variance=1)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_df)

total_data = len(dataset)
remain_data = x_scaled.shape[0]
dropped_data = len(dataset) - remain_data
print(" Total data = ", total_data, "\n", "Dropped data = ", dropped_data, "\n", "Remaining data = ", remain_data)

In [ ]:
pca = PCA(n_components=4)  # Adjust the number of components according to your requirement
x_pca = pca.fit_transform(x_scaled)

print("Original shape: {}".format(str(x_scaled.shape)))
print("Reduced shape: {}".format(str(x_pca.shape)))

In [ ]:
# https://github.com/ChawlaAvi/Daily-Dose-of-Data-Science/blob/main/Machine%20Learning/PCA-Dimensions-Hyperparameter.ipynb

def percent_formatter(x, pos):
    return f"{x*100:.0f}"

init_dimensions = 4
# Variance of individual component 
exp_var = pca.explained_variance_ratio_ 
# Cumulative variance
cumulative_explained_variance = np.cumsum(exp_var) 

fig, ax = plt.subplots(figsize=(10,8))

# Add individual variance bars
plt.bar(range(1, init_dimensions+1), exp_var, align='center',
        label='Individual Component Variance', color = '#4c72b0', edgecolor = 'black')

# Add cumulative variance line
plt.plot(range(1, init_dimensions+1), cumulative_explained_variance, marker="o",
         label='Cumulative Variance', color='#85d9b1', linewidth=2.5, markersize=8)

# Add cumulative variance annotation
for x, y in zip(range(1, init_dimensions), cumulative_explained_variance):
    ax.annotate(f"{round(y * 100):.0f}%", (x - 0.3, y + 0.03))
    
ax.annotate(f"{round(cumulative_explained_variance[3] * 100):.0f}%", (3.6, cumulative_explained_variance[3] + 0.005))

# Add label, title, ticks
plt.xlabel('Principal Components', weight='bold', fontsize = 26)
plt.ylabel('Explained Variance (%)', weight='bold', fontsize = 26)
# plt.title('Cumulative Explained Variance Plot\nfor PCA', fontsize=19, fontweight="bold", pad=20)
plt.xticks(range(1, init_dimensions+1))

# Format y-axis
ax.yaxis.set_major_formatter(mtick.FuncFormatter(percent_formatter))

# Legend location
# plt.legend(loc='best')
plt.show()

In [ ]:
cumulative_explained_variance

# PCA Transformation Eigenvector

In [ ]:
# Consider materials - Uses OneHotEncoder (encoding materials and heat treatment) without Iterative Imputer (missing values)
x_df = dataset[input_var].drop(columns=['Material'])
x_df = x_df.dropna()
x_df_index = x_df.index.to_list()

# Standardize the features (mean=0 and variance=1)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_df)

total_data = len(dataset)
remain_data = x_scaled.shape[0]
dropped_data = len(dataset) - remain_data
print(" Total data = ", total_data, "\n", "Dropped data = ", dropped_data, "\n", "Remaining data = ", remain_data)

In [ ]:
pca = PCA(n_components=2)  # Adjust the number of components according to your requirement
x_pca = pca.fit_transform(x_scaled)

print("Original shape: {}".format(str(x_scaled.shape)))
print("Reduced shape: {}".format(str(x_pca.shape)))

In [ ]:
# Eigenvector of PCA
pca.components_

In [ ]:
# https://www.kaggle.com/code/atulanandjha/dimensionality-reduction-i-pca-using-eigenvector
print('Covariance matrix \n')

cov_mat = np.cov(x_scaled.T)
eig_vals, eig_vecs = np.linalg.eig(cov_mat)
print('Eigenvectors \n%s' %cov_mat)
print('Eigenvectors \n%s' %eig_vecs)
print('\nEigenvalues \n%s' %eig_vals)